In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [2]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        if col.lower() == "species":
            rename_map[col] = "Species"
        elif col.lower() == "originlocation":
            rename_map[col] = "OriginLocation"
    return df.rename(columns=rename_map)


def get_preprocessed_df() -> pd.DataFrame:
    df = pd.read_csv("penguins.csv")
    df = normalize_columns(df)
    null_columns = df.columns[df.isnull().any()]
    numeric_cols = df.select_dtypes(include="number").columns
    for col in null_columns:
        if col in numeric_cols:
            df[col] = df.groupby("Species")[col].transform(lambda x: x.fillna(x.mean()))
    le = LabelEncoder()
    df["OriginLocation"] = le.fit_transform(df["OriginLocation"])
    return df


def split(data: pd.DataFrame):

    encoder = LabelEncoder()
    data["Species"] = encoder.fit_transform(data["Species"])

    # first_class = data[data["Species"] == -1]
    # second_class = data[data["Species"] == 1]
    # first_class= first_class.sample(frac=1).reset_index(drop=True)
    # second_class= second_class.sample(frac=1).reset_index(drop=True)
    # data = pd.concat([first_class, second_class], ignore_index=True)
    
    train_data = data.iloc[np.r_[0:30, 50:80, 100:130]]
    test_data  = data.iloc[np.r_[30:50, 80:100, 130:150]]

    sc = MinMaxScaler()
    train_data = sc.fit_transform(train_data)
    test_data  = sc.transform(test_data) 

    train_data = pd.DataFrame(train_data)
    test_data = pd.DataFrame(test_data)

    X_train = train_data.iloc[:,1:]
    y_train = train_data.iloc[:,0:1]

    X_test = test_data.iloc[:,1:]
    y_test = test_data.iloc[:,0:1]

   

    # Keep original (unscaled) test coords for the scatter plot
    test_original = test_data.copy()

    return X_train, y_train, X_test, y_test,sc


In [9]:
class Layer:
    def __init__(self, input_num, output_num, bias):
        self.bias = bias
        self.outputs = []
        self.nets = []
        self.errors = []
        self.weights = np.random.randn(input_num + self.bias, output_num) 
        # if self.bias == 0:
        #     self.biases = np.zeros((1, output_num))
        # else:
        #     self.biases = np.random.randn(1, output_num)

class mlp:
    def __init__(self,X_train ,y , hidden_layers, hidden_neurons, learning_rate, epochs, bias, activation_function):
        output_size=len(y.value_counts())
        input_size =len(X_train.columns)
        self.X_train = X_train
        self.y = y
        self.layers = []
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.hidden_layers = hidden_layers
        self.layer_num = hidden_layers+1
        self.hidden_neurons = hidden_neurons
        self.bias = bias
        self.activation_function = activation_function

        for i in range(hidden_layers + 1):
            if i == 0:
                lyr =Layer(input_size, hidden_neurons, bias)
                self.layers.append(lyr)
                print(f'Layer : {i}')
                print(lyr.weights)
            elif i == hidden_layers:
                lyr =Layer(hidden_neurons, output_size, bias)
                print('Output Layer')
                print(lyr.weights)
                self.layers.append(lyr)
            else:
                print(f'Layer : {i}')
                lyr =Layer(hidden_neurons, hidden_neurons, bias)
                print(lyr.weights)
                self.layers.append(lyr)

    def activationFn(self, actFn, net):
          e = 2.718
          if (actFn+1) == 1: # Sigmoid Function
              e = 2.718
              y = 1 /(1 + (e**net))
               
          if (actFn+1) == 2:
               y = ((e**net)-(e**(-net))) / ((e**net)+(e**(-net)))
          
          return y
      
def forward_pass(self):
    for i in range(self.layers):
        if i == 0:
            input = np.array(self.X_train.iloc[0:1,:])
        else:
            input = self.layers[i-1].outputs
        net = np.dot(input.T,self.layers[i].weights)
        # y = self.activationFn(1,net)
        self.layers[i].nets.append(net)
    print(self.nets)
        
        
        

In [4]:
df = get_preprocessed_df()

In [5]:
X_train, y_train, X_test, y_test,sc = split(df)

In [12]:
model = mlp(X_train,y_train,2,2,0.1,1,1,0)
model.forward_pass()

Layer : 0
[[-0.16990287 -0.61741971]
 [ 1.04238409  1.36362224]
 [ 1.35964523 -1.95998493]
 [ 0.20775696 -2.24950789]
 [ 0.49122613  0.14781417]
 [-0.71309808  0.1334747 ]]
Layer : 1
[[ 0.16760918 -0.61971036]
 [-0.25059447  1.39288917]
 [-0.05771137  0.16925064]]
Output Layer
[[-0.36999824  0.64434991 -0.01886529]
 [-0.32203182 -1.84982549 -1.43661837]
 [-0.94899798 -0.20496646  0.6344814 ]]


AttributeError: 'mlp' object has no attribute 'forward_pass'